# Google ADK + ChromaDB + PDF RAG

This companion notebook adds a more practical Agentic AI example to Week 3.

It shows how to build a Google ADK agent that can:

- index a **local PDF**
- index a **web PDF**
- store chunks in **ChromaDB**
- create embeddings with **Google Cloud / Vertex AI embedding models**
- answer grounded questions with **Google ADK**
- use **other tools**, such as a calculator and document catalog

The original Week 3 notebook is left unchanged.

## What You Will Learn

1. How Google ADK fits on top of a RAG pipeline.
2. How ChromaDB stores embeddings and metadata.
3. How Vertex AI embeddings are used for retrieval.
4. How to support both local-PDF and web-PDF ingestion.
5. How to expose retrieval and utility functions as ADK tools.
6. How to test the full flow end to end in a notebook.

## Architecture

The system below is intentionally simple and practical:

1. A PDF is loaded from disk or downloaded from a URL.
2. The text is extracted page by page.
3. The text is split into chunks.
4. Each chunk is embedded using Vertex AI.
5. ChromaDB stores the chunk text, metadata, and embedding vector.
6. The ADK agent calls tools to search the indexed content.
7. The final answer is written using retrieved evidence.

## Install Dependencies

In [ ]:
%pip install -q google-adk google-genai chromadb pypdf reportlab requests numpy
print('Dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.1 MB/s eta 0:00:00
Dependencies installed


## Authentication And Environment Setup

This notebook supports both Colab and local execution.

- In **Colab**, it uses built-in Google authentication.
- Locally, it uses `service-account-key.json` through `GOOGLE_APPLICATION_CREDENTIALS`.

The model backend is Vertex AI, not the Gemini API key flow.

# Update Project below

In [ ]:
# project id belongs to you
PROJECT_ID = 'dynamic-market-478415-f0'
LOCATION = 'us-central1'

In [ ]:
import os
import sys
from pathlib import Path


SERVICE_ACCOUNT_PATH = Path('../service-account-key.json')
WORK_DIR = Path('./week3_adk_chroma_assets')
DATA_DIR = WORK_DIR / 'data'
CHROMA_DIR = WORK_DIR / 'chroma_db'
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'true'
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = LOCATION

credentials = None
if 'google.colab' in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated with Google Colab user credentials')
else:
    if SERVICE_ACCOUNT_PATH.exists():
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(SERVICE_ACCOUNT_PATH.resolve())
        from google.oauth2 import service_account
        credentials = service_account.Credentials.from_service_account_file(
            SERVICE_ACCOUNT_PATH,
            scopes=['https://www.googleapis.com/auth/cloud-platform'],
        )
        print(f'Using local service account: {SERVICE_ACCOUNT_PATH.resolve()}')
    else:
        print('Running without explicit local service account; relying on ADC if available')

print(f'Project: {PROJECT_ID}')
print(f'Location: {LOCATION}')
print(f'Data directory: {DATA_DIR.resolve()}')
print(f'Chroma directory: {CHROMA_DIR.resolve()}')

Authenticated with Google Colab user credentials
Project: dynamic-market-478415-f0
Location: us-central1
Data directory: /content/week3_adk_chroma_assets/data
Chroma directory: /content/week3_adk_chroma_assets/chroma_db


## Imports And SDK Clients

We create two key clients in this notebook:

- a `genai.Client(...)` for Gemini generation and Vertex AI embeddings
- a `chromadb.PersistentClient(...)` for local vector storage

The ADK layer will sit on top of these.

In [ ]:
import asyncio
import json
import re
import uuid
from dataclasses import dataclass, field
from typing import Any, Optional

import chromadb
import numpy as np
import requests
from pypdf import PdfReader
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer
from google import genai
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

if credentials is None:
    client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
else:
    client = genai.Client(
        vertexai=True,
        project=PROJECT_ID,
        location=LOCATION,
        credentials=credentials,
    )

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
print('GenAI and Chroma clients are ready')

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


GenAI and Chroma clients are ready


In [ ]:
health = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Reply with exactly: connected',
)
probe = client.models.embed_content(
    model='text-embedding-004',
    contents=['ADK plus Chroma plus PDF RAG'],
)
print('Generation request completed')
print('Response text:', getattr(health, 'text', None))
print('Embedding dimensions:', len(probe.embeddings[0].values))

Generation request completed
Response text: connected
Embedding dimensions: 768


## Create A Sample Local PDF

To make the notebook self-contained, we create a small employee policy PDF locally.

That means the **local PDF RAG** example works even in a fresh Colab runtime.

In [ ]:
LOCAL_SAMPLE_PDF = DATA_DIR / 'sample_employee_policy.pdf'

def create_sample_policy_pdf(path: Path) -> None:
    styles = getSampleStyleSheet()
    story = [
        Paragraph('Sample Employee Policy Handbook', styles['Title']),
        Spacer(1, 12),
        Paragraph('Remote Work Policy', styles['Heading2']),
        Paragraph('Employees may work remotely up to three days per week with manager approval. Team members must be available during core collaboration hours from 10:00 AM to 3:00 PM Central Time.', styles['BodyText']),
        Spacer(1, 12),
        Paragraph('Expense Policy', styles['Heading2']),
        Paragraph('Expense reports must be submitted within 10 calendar days of the trip end date. Receipts are required for expenses of 25 dollars or more.', styles['BodyText']),
        Spacer(1, 12),
        Paragraph('Security Policy', styles['Heading2']),
        Paragraph('Employees must use the company-approved VPN when accessing confidential systems remotely and should report suspected phishing attempts within one hour.', styles['BodyText']),
    ]
    doc = SimpleDocTemplate(str(path), pagesize=letter)
    doc.build(story)

create_sample_policy_pdf(LOCAL_SAMPLE_PDF)
print(f'Created local sample PDF: {LOCAL_SAMPLE_PDF.resolve()}')

Created local sample PDF: /content/week3_adk_chroma_assets/data/sample_employee_policy.pdf


## PDF Ingestion Helpers

These helper functions do the mechanical work of RAG:

- download PDFs from the web
- extract page text
- normalize text
- split into chunks

Keeping this logic outside the agent makes the system easier to debug.

In [ ]:
WEB_SAMPLE_PDF_URL = 'https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf'

def normalize_whitespace(text: str) -> str:
    text = text.replace('\x00', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def download_pdf(url: str, destination: Path) -> Path:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    destination.write_bytes(response.content)
    return destination

def extract_pdf_pages(pdf_path: Path) -> list[dict[str, Any]]:
    reader = PdfReader(str(pdf_path))
    pages = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = normalize_whitespace(page.extract_text() or '')
        if text:
            pages.append({'page_num': page_num, 'text': text, 'pdf_name': pdf_path.name})
    return pages

def chunk_text(text: str, chunk_size: int = 800, overlap: int = 120) -> list[str]:
    text = text.strip()
    if not text:
        return []
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    chunks = []
    current = []
    for sentence in sentences:
        candidate = ' '.join(current + [sentence]).strip()
        if len(candidate) <= chunk_size or not current:
            current.append(sentence)
        else:
            chunks.append(' '.join(current).strip())
            overlap_text = ' '.join(current)[-overlap:]
            current = [f'{overlap_text} {sentence}'.strip()]
    if current:
        chunks.append(' '.join(current).strip())
    return [chunk for chunk in chunks if chunk]

## ChromaDB Store Backed By Vertex AI Embeddings

This wrapper keeps the notebook readable by hiding the lower-level collection operations.

Important idea:
- Chroma does the storage and nearest-neighbor search
- Vertex AI creates the embeddings
- metadata keeps answers traceable back to PDF name and page number

In [ ]:
@dataclass
class Chunk:
    text: str
    page_num: int
    chunk_id: int
    metadata: dict = field(default_factory=dict)

class ChromaPdfStore:
    def __init__(self, collection_name: str = 'week3_pdf_rag', embed_model: str = 'text-embedding-004'):
        self.collection_name = collection_name
        self.embed_model = embed_model
        try:
            chroma_client.delete_collection(name=self.collection_name)
        except Exception:
            pass
        self.collection = chroma_client.get_or_create_collection(name=self.collection_name)
        self.document_registry: dict[str, dict[str, Any]] = {}

    def _embed_texts(self, texts: list[str], task_type: str) -> list[list[float]]:
        response = client.models.embed_content(
            model=self.embed_model,
            contents=texts,
            config={'task_type': task_type},
        )
        return [item.values for item in response.embeddings]

    def index_pdf(self, pdf_path: Path, doc_id: str) -> dict[str, Any]:
        pages = extract_pdf_pages(pdf_path)
        chunk_objects = []
        for page in pages:
            for chunk_id, text in enumerate(chunk_text(page['text']), start=1):
                chunk_objects.append(
                    Chunk(
                        text=text,
                        page_num=page['page_num'],
                        chunk_id=chunk_id,
                        metadata={'doc_id': doc_id, 'pdf_name': page['pdf_name'], 'source': f"{page['pdf_name']}#page={page['page_num']}"},
                    )
                )
        embeddings = self._embed_texts([chunk.text for chunk in chunk_objects], task_type='RETRIEVAL_DOCUMENT')
        ids, documents, metadatas = [], [], []
        for chunk, embedding in zip(chunk_objects, embeddings):
            ids.append(f"{doc_id}-p{chunk.page_num}-c{chunk.chunk_id}")
            documents.append(chunk.text)
            metadatas.append({**chunk.metadata, 'page_num': chunk.page_num, 'chunk_id': chunk.chunk_id})
        self.collection.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=embeddings)
        self.document_registry[doc_id] = {'pdf_name': pdf_path.name, 'chunks': len(chunk_objects), 'pages': len(pages)}
        return {'status': 'success', 'doc_id': doc_id, 'pdf_name': pdf_path.name, 'pages': len(pages), 'chunks_indexed': len(chunk_objects)}

    def search(self, query: str, top_k: int = 4, doc_id: Optional[str] = None) -> list[dict[str, Any]]:
        query_embedding = self._embed_texts([query], task_type='RETRIEVAL_QUERY')[0]
        where = {'doc_id': doc_id} if doc_id else None
        result = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=max(top_k * 3, 8),
            where=where,
            include=['documents', 'metadatas', 'distances'],
        )
        query_terms = set(re.findall(r'[a-z0-9]+', query.lower()))
        ranked = []
        for doc, meta, distance in zip(result.get('documents', [[]])[0], result.get('metadatas', [[]])[0], result.get('distances', [[]])[0]):
            vector_score = max(0.0, 1.0 - float(distance))
            keyword_overlap = len(query_terms & set(re.findall(r'[a-z0-9]+', doc.lower()))) / max(1, len(query_terms))
            blended_score = (0.8 * vector_score) + (0.2 * keyword_overlap)
            ranked.append({'score': round(blended_score, 4), 'text': doc, 'metadata': meta})
        ranked.sort(key=lambda item: item['score'], reverse=True)
        return ranked[:top_k]

    def list_documents(self) -> dict[str, Any]:
        return {'status': 'success', 'documents': self.document_registry}

store = ChromaPdfStore()
print('Chroma-backed PDF store is ready')

Chroma-backed PDF store is ready


## Tool Functions For The ADK Agent

These are normal Python functions, but once we pass them into `Agent(..., tools=[...])`, the agent can decide when to call them.

The key ingredients are:

- clear function names
- readable docstrings
- simple parameter types
- structured return values

In [ ]:
def index_local_pdf(local_path: str, doc_id: Optional[str] = None) -> dict[str, Any]:
    """Index a local PDF file into ChromaDB for later semantic search.

    Args:
        local_path: Path to a PDF file on disk.
        doc_id: Optional document identifier. If omitted, one is created automatically.
    """
    path = Path(local_path)
    if not path.exists():
        return {'status': 'error', 'message': f'File not found: {local_path}'}
    resolved_doc_id = doc_id or f'local_{path.stem}'
    return store.index_pdf(path, resolved_doc_id)

def index_web_pdf(url: str, doc_id: Optional[str] = None) -> dict[str, Any]:
    """Download a PDF from the web, save it locally, and index it into ChromaDB.

    Args:
        url: Public URL to a PDF document.
        doc_id: Optional document identifier. If omitted, one is created automatically.
    """
    safe_name = re.sub(r'[^a-zA-Z0-9_-]+', '_', Path(url).stem or 'web_pdf')
    destination = DATA_DIR / f'{safe_name}.pdf'
    download_pdf(url, destination)
    resolved_doc_id = doc_id or f'web_{safe_name}'
    result = store.index_pdf(destination, resolved_doc_id)
    result['downloaded_to'] = str(destination)
    return result

def search_pdf_knowledge_base(question: str, top_k: int = 4, doc_id: Optional[str] = None) -> dict[str, Any]:
    """Search indexed PDF chunks using semantic similarity.

    Args:
        question: Natural-language search query.
        top_k: Number of chunks to return.
        doc_id: Optional document filter.
    """
    return {'status': 'success', 'question': question, 'matches': store.search(question, top_k=top_k, doc_id=doc_id)}

def list_indexed_documents() -> dict[str, Any]:
    """List PDFs that have already been indexed into ChromaDB."""
    return store.list_documents()

def calculate_expression(expression: str) -> dict[str, Any]:
    """Evaluate a basic arithmetic expression safely.

    Args:
        expression: Expression such as '18 * 12.5' or '(150-50)/5'.
    """
    allowed = set('0123456789+-*/(). %')
    if any(ch not in allowed for ch in expression):
        return {'status': 'error', 'message': 'Only basic arithmetic characters are allowed'}
    try:
        value = eval(expression, {'__builtins__': {}}, {})
        return {'status': 'success', 'expression': expression, 'result': value}
    except Exception as exc:
        return {'status': 'error', 'message': str(exc)}

print('Tool functions are ready')

Tool functions are ready


## Direct Tool Examples

Before giving tools to the ADK agent, it is useful to test them directly. That helps separate retrieval bugs from agent-orchestration bugs.

In [ ]:
local_index_result = index_local_pdf(str(LOCAL_SAMPLE_PDF), doc_id='sample_policy')
print(json.dumps(local_index_result, indent=2))

{
  "status": "success",
  "doc_id": "sample_policy",
  "pdf_name": "sample_employee_policy.pdf",
  "pages": 1,
  "chunks_indexed": 1
}


In [ ]:
web_index_result = index_web_pdf(WEB_SAMPLE_PDF_URL, doc_id='dummy_web_pdf')
print(json.dumps(web_index_result, indent=2))

{
  "status": "success",
  "doc_id": "dummy_web_pdf",
  "pdf_name": "dummy.pdf",
  "pages": 1,
  "chunks_indexed": 1,
  "downloaded_to": "week3_adk_chroma_assets/data/dummy.pdf"
}


In [ ]:
print(json.dumps(list_indexed_documents(), indent=2))

{
  "status": "success",
  "documents": {
    "sample_policy": {
      "pdf_name": "sample_employee_policy.pdf",
      "chunks": 1,
      "pages": 1
    },
    "dummy_web_pdf": {
      "pdf_name": "dummy.pdf",
      "chunks": 1,
      "pages": 1
    }
  }
}


In [ ]:
print(json.dumps(search_pdf_knowledge_base('What are the remote work rules?', top_k=3, doc_id='sample_policy'), indent=2))

{
  "status": "success",
  "question": "What are the remote work rules?",
  "matches": [
    {
      "score": 0.4152,
      "text": "Sample Employee Policy Handbook Remote Work Policy Employees may work remotely up to three days per week with manager approval. Team members must be available during core collaboration hours from 10:00 AM to 3:00 PM Central Time. Expense Policy Expense reports must be submitted within 10 calendar days of the trip end date. Receipts are required for expenses of 25 dollars or more. Security Policy Employees must use the company-approved VPN when accessing confidential systems remotely and should report suspected phishing attempts within one hour.",
      "metadata": {
        "chunk_id": 1,
        "doc_id": "sample_policy",
        "pdf_name": "sample_employee_policy.pdf",
        "page_num": 1,
        "source": "sample_employee_policy.pdf#page=1"
      }
    }
  ]
}


## Build The Google ADK Agent

The ADK agent below gets a small set of tools:

- two indexing tools
- one semantic search tool
- one listing tool
- one calculator tool

The instruction forces it to use retrieval before answering PDF-related questions.

In [ ]:
agent = Agent(
    name='pdf_rag_assistant',
    model='gemini-2.5-flash',
    instruction=(
        'You are a helpful PDF research assistant. '
        'For questions about indexed documents, always call search_pdf_knowledge_base before answering. '
        'Ground your answer in the returned evidence and cite the source field. '
        'If a user asks to index a PDF, use index_local_pdf or index_web_pdf. '
        'Use list_indexed_documents to inspect what is available. '
        'Use calculate_expression for arithmetic questions.'
    ),
    tools=[
        index_local_pdf,
        index_web_pdf,
        search_pdf_knowledge_base,
        list_indexed_documents,
        calculate_expression,
    ],
)

runner = InMemoryRunner(agent=agent, app_name='week3_adk_chroma_rag')
print('ADK agent and runner are ready')

ADK agent and runner are ready


In [ ]:
async def ask_agent(question: str, user_id: str = 'student1', session_id: str = 'week3-demo') -> str:
    events = await runner.run_debug(question, user_id=user_id, session_id=session_id, quiet=True)
    final_text = ''
    for event in events:
        if event.is_final_response() and getattr(event, 'content', None):
            texts = [part.text for part in event.content.parts if getattr(part, 'text', None)]
            if texts:
                final_text = '\n'.join(texts)
    return final_text

print('Notebook helper `ask_agent(...)` is ready')

Notebook helper `ask_agent(...)` is ready


## ADK Agent Examples

These final cells demonstrate the agent using PDF RAG and one extra utility tool.

In [ ]:
print(await ask_agent('List the indexed documents we currently have.'))

The indexed documents are:
* `dummy_web_pdf` (1 page, 1 chunk) - dummy.pdf
* `sample_policy` (1 page, 1 chunk) - sample_employee_policy.pdf


In [ ]:
print(await ask_agent('What are the remote work rules in the sample employee policy PDF?'))

The remote work policy in the sample employee policy PDF states that:
* Employees may work remotely up to three days per week with manager approval.
* Team members must be available during core collaboration hours from 10:00 AM to 3:00 PM Central Time.

Source: sample_employee_policy.pdf#page=1


In [ ]:
print(await ask_agent('What does the dummy web PDF say?'))

The dummy web PDF says "Dummy PDF file".

Source: dummy.pdf#page=1


In [ ]:
print(await ask_agent('Calculate 18 * 12.5 and tell me the result.'))

The result of 18 * 12.5 is 225.


In [ ]:
print(await ask_agent('ingest the wed pdf given in the url "https://www.princexml.com/samples/newsletter/drylab.pdf"'))

The PDF from the URL "https://www.princexml.com/samples/newsletter/drylab.pdf" has been successfully indexed with the document ID "drylab_newsletter". It contains 3 pages and 11 chunks.


In [ ]:
print(await ask_agent("give me infomraiton on drylab"))

Drylab is preparing for the launch of Drylab 3.0 at the International Broadcasters Convention in Amsterdam in September. They are working to get solid feedback from pilot users before the launch. Drylab's Annual General Meeting will be held on June 16th at 15:00. They are also working towards a US launch with Drylab 3.0, while maintaining their existing system in Europe.

Drylab also successfully completed an investment round, raising 2.13 MNOK to match a 2.05 MNOK loan from Innovation Norway. Including a development agreement, the total new capital is 5 MNOK. They have new owners: Unni Jacobsen, Torstein Jahr, Suzanne Bolstad, Eivind Bergene, Turid Brun, Vigdis Trondsen, Lea Blindheim, and Kristine.

Sources:
* drylab.pdf#page=3
* drylab.pdf#page=1


In [ ]:
print(await ask_agent('ingest the local pdf given in the path "/content/week3_adk_chroma_assets/data/drylab_local.pdf"'))

The local PDF from the path "/content/week3_adk_chroma_assets/data/drylab_local.pdf" has been successfully indexed with the document ID "drylab_local". It contains 3 pages and 11 chunks.


In [ ]:
print(await ask_agent("give me infomraiton on drylab"))

Drylab is currently experiencing a one-month delay in development but anticipates this to decrease with new developers on board. The launch of Drylab 3.0 is scheduled for September at the International Broadcasters Convention in Amsterdam, and they are actively seeking feedback from pilot users. Their Annual General Meeting (AGM) will be held on June 16th at 15:00. Drylab is also working towards a US launch for Drylab 3.0 and maintaining its existing system in Europe.

In terms of finances and ownership, Drylab successfully closed an investment round, raising 2.13 MNOK, which, combined with a 2.05 MNOK loan from Innovation Norway and a development agreement with Filmlance International, brings their total new capital to 5 MNOK. They have welcomed new owners, including Unni Jacobsen, Torstein Jahr, Suzanne Bolstad, Eivind Bergene, Turid Brun, Vigdis Trondsen, Lea Blindheim, and Kristine.

Sources:
* drylab.pdf#page=3
* drylab.pdf#page=1
* drylab_local.pdf#page=3
* drylab_local.pdf#page=

Multi-Tool Research Agent with RAG with FAISS

Let's build a comprehensive RAG agent that:
- Downloads research papers/documents from the internet
- Extracts and indexes content from PDFs and CSV files
- Performs web searches for up-to-date information
- Includes a calculator for quantitative analysis
- Combines all tools to answer complex research questions

### ✅ Google ADK Integration Overview

**Complete Stack:**
```
┌─────────────────────────────────────────┐
│     Google ADK Agent Framework          │
│  (Agent, Runner, SessionService)        │
└─────────────────┬───────────────────────┘
                  │
        ┌─────────┴─────────┐
        │                   │
┌───────▼──────┐   ┌────────▼────────┐
│  Gemini 2.5  │   │   Tool Layer    │
│  Flash LLM   │   │  (Your Tools)   │
└──────────────┘   └────────┬────────┘
                            │
        ┌───────────────────┼───────────────────┐
        │                   │                   │
┌───────▼──────┐   ┌────────▼────────┐  ┌──────▼──────┐
│  Vertex AI   │   │   FAISS Vector  │  │   Local     │
│  Embeddings  │   │     Search      │  │ File System │
└──────────────┘   └─────────────────┘  └─────────────┘
```

**How it Works:**

1. **User Query** → ADK Agent
2. **Agent** decides which tool to call
3. **Tool** uses Vertex AI + FAISS
4. **Results** return to Agent
5. **Agent** generates response with LLM

✅ 100% Google ADK compatible  
✅ Uses ADK's tool calling system  
✅ Works with ADK sessions  
✅ Integrates with Vertex AI  
✅ Local file system (no cloud dependencies)

In [ ]:
# Install additional packages for RAG with FAISS and embeddings
!pip install -q requests beautifulsoup4 lxml PyPDF2 pandas numpy faiss-cpu
!pip install -q google-cloud-aiplatform

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.5 MB/s eta 0:00:00


In [ ]:
    # ============================================================================
    # IMPORTS FOR RAG WITH FAISS AND VERTEX AI
    # ============================================================================

    # Web scraping and file downloading
    import requests                          # HTTP library for downloading files from URLs
    from bs4 import BeautifulSoup           # HTML parser for extracting text from web pages
    import PyPDF2                            # PDF reader for extracting text from PDF files
    import pandas as pd                      # Data manipulation for CSV files
    import numpy as np                       # Numerical operations and array handling
    import re                                # Regular expressions for text processing
    from urllib.parse import urlparse        # URL parsing utilities
    import io                                # File-like object operations

    # Vector search and embeddings
    import faiss                             # Facebook AI Similarity Search - fast vector search
    from vertexai.language_models import TextEmbeddingModel, TextEmbeddingInput
                                            # Vertex AI embedding models for semantic search

    # ============================================================================
    # INITIALIZE VERTEX AI EMBEDDING MODEL
    # ============================================================================
    # text-embedding-005: Latest Vertex AI model
    # - 768 dimensions
    # - Optimized for semantic similarity
    # - Supports up to ~20k tokens per request
    embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-005")

    # ============================================================================
    # GLOBAL STORAGE FOR RAG SYSTEM
    # ============================================================================
    from typing import Dict
    # Main document storage: {doc_id: {content, chunks, embedding, metadata, indexed_at}}
    document_store: Dict[str, Dict[str, Any]] = {}

    # FAISS index for vector similarity search
    # Will be initialized as IndexFlatL2 when first document is added
    faiss_index = None

    # Bidirectional mappings between doc_ids and FAISS indices
    doc_id_to_idx: Dict[str, int] = {}  # Maps doc_id → FAISS index position (e.g., "doc_1" → 0)
    idx_to_doc_id: Dict[int, str] = {}  # Maps FAISS index position → doc_id (e.g., 0 → "doc_1")

    print("✓ Imported libraries for RAG with FAISS and Vertex AI embeddings")
    print("✓ Initialized Vertex AI text-embedding-005 model (768 dimensions)")
    print("✓ Created global storage: document_store, faiss_index, mappings")

✓ Imported libraries for RAG with FAISS and Vertex AI embeddings
✓ Initialized Vertex AI text-embedding-005 model (768 dimensions)
✓ Created global storage: document_store, faiss_index, mappings


/usr/local/lib/python3.12/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [ ]:
# Import required libraries
import os
import asyncio
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from typing import Dict, Any, List, Optional
from datetime import datetime
import json

print("✓ All imports successful!")

✓ All imports successful!


In [ ]:
# ============================================================================
# TOOL 1: DOWNLOAD AND INDEX FILES WITH VECTOR EMBEDDINGS
# ============================================================================

def get_embedding(text: str) -> np.ndarray:
    """
    Generate semantic embedding vector for text using Vertex AI.

    How it works:
    1. Takes input text (string)
    2. Calls Vertex AI text-embedding-005 model
    3. Returns 768-dimensional vector representing semantic meaning

    Args:
        text: Text to embed (max ~20k tokens, we truncate at 10k chars for safety)

    Returns:
        Numpy array of shape (768,) with float32 values

    Example:
        embedding = get_embedding("The cat sat on the mat")
        # Returns: array([0.123, -0.456, 0.789, ...], shape=(768,))
    """
    # Truncate text if too long to avoid API limits
    max_chars = 10000  # Conservative limit (~2500 tokens)
    if len(text) > max_chars:
        text = text[:max_chars]

    # Call Vertex AI embedding API
    embeddings = embedding_model.get_embeddings([text])

    # Convert to numpy array for FAISS compatibility
    return np.array(embeddings[0].values, dtype='float32')


def download_and_index_file(url: str, doc_id: Optional[str] = None) -> Dict[str, Any]:
    """
    Download a file from the internet and index it with vector embeddings for semantic search.

    Complete workflow:
    1. Download file from URL (PDF, CSV, HTML, or text)
    2. Extract and parse content based on file type
    3. Generate semantic embedding using Vertex AI
    4. Add embedding to FAISS index for fast similarity search
    5. Store document in local document store

    Supported file types:
    - PDF: Extracts text from all pages
    - CSV: Creates searchable summary with statistics
    - HTML: Extracts clean text (removes scripts/styles)
    - Text: Stores as-is

    Args:
        url: Full URL of the file to download (e.g., "https://arxiv.org/pdf/1706.03762.pdf")
        doc_id: Optional custom identifier (auto-generated as "doc_1", "doc_2", etc. if not provided)

    Returns:
        Dictionary with:
        - success: True/False
        - doc_id: Document identifier
        - type: File type (pdf, csv, html, text)
        - content_length: Number of characters
        - chunks_count: Number of chunks created
        - metadata: Additional file information
        - message: Success/error message

    Example:
        result = download_and_index_file(
            url="https://arxiv.org/pdf/1706.03762.pdf",
            doc_id="transformer_paper"
        )
        # Downloads paper, generates embedding, adds to FAISS index
    """
    global faiss_index, doc_id_to_idx, idx_to_doc_id  # Access global RAG storage

    try:
        print(f"  [Tool] Downloading file from: {url}")

        # ====================================================================
        # STEP 1: DOWNLOAD FILE
        # ====================================================================
        response = requests.get(
            url,
            timeout=30,  # 30 second timeout
            headers={'User-Agent': 'Mozilla/5.0'}  # Pretend to be a browser
        )
        response.raise_for_status()  # Raise exception for 4xx/5xx errors

        # ====================================================================
        # STEP 2: GENERATE DOC_ID
        # ====================================================================
        if not doc_id:
            doc_id = f"doc_{len(document_store) + 1}"  # Auto-increment: doc_1, doc_2, ...

        # ====================================================================
        # STEP 3: DETECT FILE TYPE
        # ====================================================================
        content_type = response.headers.get('content-type', '').lower()
        parsed_url = urlparse(url)
        file_extension = parsed_url.path.split('.')[-1].lower()
        print("I'm here in detect file")
        # Initialize document data
        document_content = ""
        metadata = {"url": url, "doc_id": doc_id, "type": "unknown"}
        chunks = []  # Store chunks for fine-grained retrieval

        # ====================================================================
        # STEP 4: PROCESS PDF FILES
        # ====================================================================
        if 'pdf' in content_type or file_extension == 'pdf':
            print(f"  [Tool] Processing PDF file...")

            # Convert downloaded bytes to file-like object
            pdf_file = io.BytesIO(response.content)
            pdf_reader = PyPDF2.PdfReader(pdf_file)

            pages_text = []
            # Extract text from each page
            for page_num, page in enumerate(pdf_reader.pages):
                try:
                    page_text = page.extract_text()
                    pages_text.append({
                        "page": page_num + 1,
                        "text": page_text
                    })

                    # Create a chunk for each page (useful for page-level retrieval)
                    chunks.append({
                        "text": page_text,
                        "metadata": {"page": page_num + 1}
                    })

                    # Concatenate all page text
                    document_content += f"\\n[Page {page_num + 1}]\\n{page_text}\\n"

                except Exception as e:
                    print(f"  [Warning] Could not extract page {page_num + 1}: {e}")

            # Update metadata with PDF-specific info
            metadata.update({
                "type": "pdf",
                "pages": len(pages_text),
                "pages_data": pages_text
            })

        # ====================================================================
        # STEP 5: PROCESS CSV FILES
        # ====================================================================
        elif 'csv' in content_type or file_extension == 'csv':
            print(f"  [Tool] Processing CSV file...")

            # Parse CSV into pandas DataFrame
            csv_content = io.StringIO(response.text)
            df = pd.read_csv(csv_content)

            # Create searchable text summary of CSV
            document_content = f"CSV Data Summary:\\n"
            document_content += f"Columns: {', '.join(df.columns)}\\n"
            document_content += f"Rows: {len(df)}\\n\\n"
            document_content += f"Data Preview:\\n{df.head(10).to_string()}\\n\\n"
            document_content += f"Statistics:\\n{df.describe().to_string()}"

            # Create single chunk for CSV (entire summary)
            chunks.append({
                "text": document_content,
                "metadata": {"type": "csv_summary"}
            })

            # Update metadata with CSV-specific info
            metadata.update({
                "type": "csv",
                "columns": list(df.columns),
                "row_count": len(df),
                "dataframe_summary": df.describe().to_dict()
            })

        # ====================================================================
        # STEP 6: PROCESS TEXT/HTML FILES
        # ====================================================================
        else:
            print(f"  [Tool] Processing text/HTML content...")

            if 'html' in content_type:
                # Parse HTML and extract clean text
                soup = BeautifulSoup(response.text, 'html.parser')

                # Remove script and style elements
                for script in soup(["script", "style"]):
                    script.decompose()

                # Extract text with line breaks
                document_content = soup.get_text(separator='\\n', strip=True)
                metadata["type"] = "html"
            else:
                # Plain text file
                document_content = response.text
                metadata["type"] = "text"

            # Create single chunk for text
            chunks.append({
                "text": document_content,
                "metadata": {"type": metadata["type"]}
            })

        # ====================================================================
        # STEP 7: GENERATE EMBEDDING WITH VERTEX AI
        # ====================================================================
        print(f"  [Tool] Generating semantic embedding with Vertex AI...")
        embedding = get_embedding(document_content)  # Returns 768-dim vector

        # ====================================================================
        # STEP 8: INITIALIZE FAISS INDEX (FIRST TIME ONLY)
        # ====================================================================
        if faiss_index is None:
            dimension = len(embedding)  # Should be 768 for text-embedding-005

            # Create IndexFlatL2: Exact search using L2 (Euclidean) distance
            # - "Flat" = exhaustive search (checks all vectors)
            # - "L2" = Euclidean distance metric
            faiss_index = faiss.IndexFlatL2(dimension)

            print(f"  [Tool] Initialized FAISS IndexFlatL2 with dimension {dimension}")

        # ====================================================================
        # STEP 9: ADD EMBEDDING TO FAISS INDEX
        # ====================================================================
        idx = len(doc_id_to_idx)  # Current index position (0, 1, 2, ...)

        # Add embedding to FAISS (requires shape: (1, 768))
        faiss_index.add(embedding.reshape(1, -1))

        # Create bidirectional mappings
        doc_id_to_idx[doc_id] = idx      # "doc_1" → 0
        idx_to_doc_id[idx] = doc_id      # 0 → "doc_1"

        # ====================================================================
        # STEP 10: STORE DOCUMENT IN DOCUMENT STORE
        # ====================================================================
        document_store[doc_id] = {
            "content": document_content,      # Full text content
            "chunks": chunks,                 # List of chunks with metadata
            "embedding": embedding,           # 768-dim vector
            "metadata": metadata,             # File type, URL, etc.
            "indexed_at": datetime.now().isoformat()  # Timestamp
        }

        print(f"  [Tool] Successfully indexed document '{doc_id}' ({len(document_content)} characters)")
        print(f"  [Tool] Added to FAISS index at position {idx}")

        # ====================================================================
        # RETURN SUCCESS RESULT
        # ====================================================================
        return {
            "success": True,
            "doc_id": doc_id,
            "type": metadata["type"],
            "content_length": len(document_content),
            "chunks_count": len(chunks),
            "metadata": metadata,
            "message": f"Successfully downloaded and indexed {metadata['type']} file as '{doc_id}' with vector embeddings"
        }

    except requests.exceptions.RequestException as e:
        # Handle download errors (timeout, connection error, etc.)
        print(f"  [Error] Download failed: {e}")
        return {
            "success": False,
            "error": f"Failed to download file: {str(e)}"
        }

    except Exception as e:
        # Handle any other errors (parsing, embedding, etc.)
        print(f"  [Error] Processing failed: {e}")
        return {
            "success": False,
            "error": f"Failed to process file: {str(e)}"
        }

print("✓ Created file download and vector indexing tool with Vertex AI embeddings")
print("  - Supports: PDF, CSV, HTML, text files")
print("  - Generates: 768-dim semantic embeddings")
print("  - Indexes: Adds to FAISS for fast similarity search")

✓ Created file download and vector indexing tool with Vertex AI embeddings
  - Supports: PDF, CSV, HTML, text files
  - Generates: 768-dim semantic embeddings
  - Indexes: Adds to FAISS for fast similarity search


In [ ]:
# ============================================================================
# TOOL 2: SEMANTIC DOCUMENT SEARCH WITH FAISS VECTOR SIMILARITY
# ============================================================================

def search_documents(query: str, doc_id: Optional[str] = None, top_k: int = 3) -> Dict[str, Any]:
    """
    Search through downloaded documents using semantic vector similarity (not keyword matching!).

    How semantic search works:
    1. Convert user query to 768-dim embedding vector (represents meaning)
    2. Use FAISS to find documents with similar embedding vectors (k-NN search)
    3. Return most semantically similar documents, ranked by similarity score

    Why this is better than keyword search:
    - Finds documents by MEANING, not just exact word matches
    - Understands synonyms: "automobile" matches "car"
    - Understands context: "bank" (financial) vs "bank" (river)
    - Works across languages (with multilingual models)

    Example:
        Query: "neural network architecture"
        Will find: "deep learning models", "transformer design", etc.
        Even if they don't contain the exact words!

    Args:
        query: Natural language search query (e.g., "What is attention mechanism?")
        doc_id: Optional - search only this specific document (None = search all docs)
        top_k: Number of most similar documents to return (default: 3)

    Returns:
        Dictionary with:
        - success: True/False
        - query: Original search query
        - results: List of matching documents with:
            * rank: Position in results (1, 2, 3, ...)
            * doc_id: Document identifier
            * similarity_score: 0-1 (higher = more similar)
            * distance: L2 distance from FAISS (lower = more similar)
            * metadata: File info, URL, etc.
            * relevant_excerpts: Text snippets containing query terms
        - count: Number of results returned
        - search_method: "faiss_vector_search" or "vector_similarity"
        - total_docs_in_index: Total documents in FAISS

    Example:
        results = search_documents("transformer attention mechanism", top_k=5)
        # Returns 5 most semantically similar documents, even if they use
        # different words like "self-attention layers" or "neural architecture"
    """
    try:
        print(f"  [Tool] Performing vector search for: '{query}'")

        # ====================================================================
        # VALIDATION: CHECK IF DOCUMENTS EXIST
        # ====================================================================
        if not document_store:
            return {
                "success": False,
                "error": "No documents have been downloaded yet. Use download_and_index_file first."
            }

        if faiss_index is None or faiss_index.ntotal == 0:
            return {
                "success": False,
                "error": "FAISS index is empty. Please download and index documents first."
            }

        # ====================================================================
        # STEP 1: GENERATE QUERY EMBEDDING
        # ====================================================================
        # Convert query text to 768-dim vector using same embedding model
        # This ensures query and documents are in the same semantic space
        print(f"  [Tool] Generating query embedding...")
        query_embedding = get_embedding(query)

        # ====================================================================
        # CASE A: SEARCH SPECIFIC DOCUMENT
        # ====================================================================
        if doc_id:
            # User specified a document to search within
            if doc_id not in document_store:
                return {
                    "success": False,
                    "error": f"Document '{doc_id}' not found"
                }

            # Get document data
            doc_data = document_store[doc_id]
            doc_embedding = doc_data["embedding"]

            # Compute similarity using L2 distance
            # L2 distance = sqrt(sum((a - b)^2))
            # Lower distance = more similar vectors
            distance = np.linalg.norm(query_embedding - doc_embedding)

            # Convert distance to similarity score (0-1 range, higher is better)
            # Formula: 1 / (1 + distance)
            # - distance=0 → similarity=1.0 (perfect match)
            # - distance=∞ → similarity=0.0 (completely different)
            similarity = 1 / (1 + distance)

            # Find relevant text excerpts (keyword-based, for display only)
            lines = doc_data["content"].split('\\n')
            relevant_lines = []
            query_terms = set(query.lower().split())

            for line in lines:
                if any(term in line.lower() for term in query_terms):
                    relevant_lines.append(line.strip())
                    if len(relevant_lines) >= 5:
                        break

            # Build result for single document
            results = [{
                "doc_id": doc_id,
                "similarity_score": float(similarity),
                "distance": float(distance),
                "metadata": doc_data["metadata"],
                "relevant_excerpts": relevant_lines[:5],
                "indexed_at": doc_data["indexed_at"]
            }]

            print(f"  [Tool] Found document with similarity score: {similarity:.4f}")

            return {
                "success": True,
                "query": query,
                "results": results,
                "count": 1,
                "search_method": "vector_similarity"
            }

        # ====================================================================
        # CASE B: SEARCH ALL DOCUMENTS WITH FAISS
        # ====================================================================
        print(f"  [Tool] Searching FAISS index with {faiss_index.ntotal} documents...")

        # Perform k-Nearest Neighbors (k-NN) search
        # FAISS.search() returns:
        # - distances: L2 distances to k nearest neighbors
        # - indices: FAISS index positions of k nearest neighbors
        k = min(top_k, faiss_index.ntotal)  # Don't ask for more results than we have

        # Query must be shape (1, 768) for FAISS
        distances, indices = faiss_index.search(query_embedding.reshape(1, -1), k)

        # ====================================================================
        # STEP 2: BUILD RESULTS FROM FAISS OUTPUT
        # ====================================================================
        results = []

        # Iterate through FAISS results
        # distances[0] = [dist1, dist2, dist3, ...]
        # indices[0] = [idx1, idx2, idx3, ...]
        for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            if idx == -1:  # FAISS returns -1 for empty slots
                continue

            # Map FAISS index back to doc_id
            doc_id_result = idx_to_doc_id[idx]
            doc_data = document_store[doc_id_result]

            # Convert L2 distance to similarity score (0-1 range)
            similarity = 1 / (1 + float(dist))

            # Find relevant text excerpts for display
            # This is keyword-based matching, separate from vector search
            lines = doc_data["content"].split('\\n')
            relevant_lines = []
            query_terms = set(query.lower().split())

            for line in lines:
                if any(term in line.lower() for term in query_terms):
                    relevant_lines.append(line.strip())
                    if len(relevant_lines) >= 5:
                        break

            # Build result entry
            results.append({
                "rank": i + 1,                        # 1st, 2nd, 3rd, etc.
                "doc_id": doc_id_result,              # Document identifier
                "similarity_score": similarity,        # 0-1 score (higher = better)
                "distance": float(dist),              # Raw L2 distance
                "metadata": doc_data["metadata"],     # File type, URL, etc.
                "relevant_excerpts": relevant_lines[:5] if relevant_lines else ["No keyword matches found"],
                "indexed_at": doc_data["indexed_at"] # When it was indexed
            })

        # ====================================================================
        # STEP 3: LOG RESULTS
        # ====================================================================
        print(f"  [Tool] Found {len(results)} relevant documents using vector search")
        for r in results:
            print(f"    - {r['doc_id']}: similarity={r['similarity_score']:.4f}, distance={r['distance']:.4f}")

        # ====================================================================
        # RETURN RESULTS
        # ====================================================================
        return {
            "success": True,
            "query": query,
            "results": results,
            "count": len(results),
            "search_method": "faiss_vector_search",
            "total_docs_in_index": faiss_index.ntotal
        }

    except Exception as e:
        # Handle any errors during search
        print(f"  [Error] Search failed: {e}")
        import traceback
        traceback.print_exc()
        return {
            "success": False,
            "error": f"Search failed: {str(e)}"
        }

print("✓ Created vector-based document search tool with FAISS")
print("  - Uses: Semantic similarity (not keyword matching)")
print("  - Algorithm: k-Nearest Neighbors (k-NN) with L2 distance")
print("  - Returns: Documents ranked by similarity score (0-1)")

✓ Created vector-based document search tool with FAISS
  - Uses: Semantic similarity (not keyword matching)
  - Algorithm: k-Nearest Neighbors (k-NN) with L2 distance
  - Returns: Documents ranked by similarity score (0-1)


In [ ]:
# ============================================================================
# TOOL 3: WEB SEARCH (Simulated for Demo)
# ============================================================================

def web_search(query: str, num_results: int = 5) -> Dict[str, Any]:
    """
    Search the web for current information (simulated for demo purposes).

    Purpose:
    - Find up-to-date information not in downloaded documents
    - Get current news, recent developments, live data
    - Complement RAG with web knowledge

    Current implementation:
    - Returns SIMULATED results for demonstration
    - Shows the structure and format expected by the agent

    Production implementation:
    - Replace with Google Custom Search API
    - Or use services like Serper API, Tavily, etc.
    - Requires API key and configuration

    Args:
        query: Search query (e.g., "latest AI developments 2025")
        num_results: Number of results to return (default: 5)

    Returns:
        Dictionary with:
        - success: True/False
        - query: Original search query
        - results: List of search results with:
            * title: Page title
            * url: Full URL
            * snippet: Text excerpt/description
            * source: Website name
        - count: Number of results
        - note: Reminder that this is simulated

    Example production integration:
        ```python
        # Using Google Custom Search API
        from googleapiclient.discovery import build

        api_key = os.environ["GOOGLE_SEARCH_API_KEY"]
        cse_id = os.environ["GOOGLE_CSE_ID"]

        service = build("customsearch", "v1", developerKey=api_key)
        result = service.cse().list(q=query, cx=cse_id, num=num_results).execute()
        return result['items']
        ```
    """
    try:
        print(f"  [Tool] Searching web for: '{query}'")

        # ====================================================================
        # SIMULATED WEB SEARCH RESULTS
        # ====================================================================
        # In production, replace this with actual API calls to:
        # - Google Custom Search API (search.googleapis.com)
        # - Serper API (serper.dev)
        # - Tavily Search API (tavily.com)
        # - Bing Web Search API (azure.microsoft.com)

        simulated_results = [
            {
                "title": f"Recent findings on {query}",
                "url": f"https://example.com/article-1",
                "snippet": f"Latest research and insights about {query}. This article covers the most recent developments and breakthrough discoveries in the field...",
                "source": "Research Journal"
            },
            {
                "title": f"{query}: A comprehensive guide",
                "url": f"https://example.com/guide",
                "snippet": f"Everything you need to know about {query}, including best practices, expert recommendations, and practical applications...",
                "source": "Expert Blog"
            },
            {
                "title": f"Understanding {query} in 2025",
                "url": f"https://example.com/2025-trends",
                "snippet": f"The current state of {query} and future predictions. Industry experts weigh in on emerging trends and what to expect...",
                "source": "Industry News"
            },
            {
                "title": f"Top 10 facts about {query}",
                "url": f"https://example.com/facts",
                "snippet": f"Discover surprising facts and interesting insights about {query} that you may not have known before...",
                "source": "Education Site"
            },
            {
                "title": f"How to apply {query} in practice",
                "url": f"https://example.com/tutorial",
                "snippet": f"Step-by-step tutorial on implementing {query} in example scenarios with code examples and case studies...",
                "source": "Tutorial Platform"
            }
        ]

        # Return requested number of results
        results = simulated_results[:num_results]

        print(f"  [Tool] Found {len(results)} web results (simulated)")

        # ====================================================================
        # RETURN SIMULATED RESULTS
        # ====================================================================
        return {
            "success": True,
            "query": query,
            "results": results,
            "count": len(results),
            "note": "⚠️ Using simulated results. In production, integrate with real search API."
        }

    except Exception as e:
        print(f"  [Error] Web search failed: {e}")
        return {
            "success": False,
            "error": f"Web search failed: {str(e)}"
        }

print("✓ Created web search tool (simulated)")
print("  - Current: Returns simulated results for demo")
print("  - Production: Replace with Google Custom Search API or similar")
print("  - Purpose: Complement RAG with current web information")

✓ Created web search tool (simulated)
  - Current: Returns simulated results for demo
  - Production: Replace with Google Custom Search API or similar
  - Purpose: Complement RAG with current web information


In [ ]:
# ============================================================================
# TOOL 4: CALCULATOR FOR QUANTITATIVE ANALYSIS
# ============================================================================

def calculate(expression: str) -> Dict[str, Any]:
    """
    Evaluate mathematical expressions and perform calculations safely.

    Purpose:
    - Perform mathematical computations
    - Calculate statistics (mean, median, std, etc.)
    - Evaluate formulas and equations
    - Support scientific functions (sqrt, log, sin, cos, etc.)

    Security:
    - Uses RESTRICTED evaluation (not full Python eval)
    - Only allows whitelisted math functions
    - No access to file system, network, or dangerous operations
    - Safe for user input

    Supported operations:
    - Basic: +, -, *, /, ** (power), % (modulo)
    - Functions: sqrt, log, log10, exp, sin, cos, tan, abs, round
    - Statistics: sum, mean, median, std, min, max
    - Constants: pi, e

    Args:
        expression: Mathematical expression as string
                   Examples:
                   - "2 + 2"
                   - "sqrt(16)"
                   - "log(100)"
                   - "(50 - 32) * 5 / 9"  (Fahrenheit to Celsius)
                   - "mean([1, 2, 3, 4, 5])"

    Returns:
        Dictionary with:
        - success: True/False
        - expression: Original expression
        - result: Computed result (as float)
        - type: Data type of result
        - error: Error message (if failed)
        - help: Usage hint (if failed)

    Examples:
        calculate("2 + 2")           # Returns: 4.0
        calculate("sqrt(16)")        # Returns: 4.0
        calculate("pi * 2")          # Returns: 6.283185307179586
        calculate("log10(1000)")     # Returns: 3.0
        calculate("mean([1,2,3])")   # Returns: 2.0
    """
    try:
        print(f"  [Tool] Calculating: {expression}")

        # ====================================================================
        # DEFINE ALLOWED FUNCTIONS (WHITELIST FOR SECURITY)
        # ====================================================================
        # Only these functions can be used in expressions
        # This prevents malicious code execution
        allowed_names = {
            # Numpy math functions
            'sqrt': np.sqrt,         # Square root: sqrt(16) = 4
            'log': np.log,           # Natural log: log(e) = 1
            'log10': np.log10,       # Base-10 log: log10(100) = 2
            'exp': np.exp,           # Exponential: exp(1) = e
            'sin': np.sin,           # Sine: sin(pi/2) = 1
            'cos': np.cos,           # Cosine: cos(0) = 1
            'tan': np.tan,           # Tangent: tan(pi/4) = 1

            # Python built-in math functions
            'abs': abs,              # Absolute value: abs(-5) = 5
            'round': round,          # Round: round(3.7) = 4
            'sum': sum,              # Sum list: sum([1,2,3]) = 6
            'min': min,              # Minimum: min([1,2,3]) = 1
            'max': max,              # Maximum: max([1,2,3]) = 3

            # Numpy statistics
            'mean': np.mean,         # Average: mean([1,2,3]) = 2
            'median': np.median,     # Median: median([1,2,3]) = 2
            'std': np.std,           # Standard deviation

            # Mathematical constants
            'pi': np.pi,             # π ≈ 3.14159
            'e': np.e                # e ≈ 2.71828
        }

        # ====================================================================
        # CLEAN AND PREPARE EXPRESSION
        # ====================================================================
        expression = expression.strip()  # Remove leading/trailing whitespace

        # ====================================================================
        # SAFELY EVALUATE EXPRESSION
        # ====================================================================
        # eval() with restricted namespace:
        # - "__builtins__": {} → Disables built-in functions (print, open, etc.)
        # - allowed_names → Only allows whitelisted functions
        # This prevents dangerous operations like:
        # - File access: open(), read(), write()
        # - Network: requests, urllib
        # - System: os.system(), subprocess
        # - Import: __import__()
        result = eval(expression, {"__builtins__": {}}, allowed_names)

        print(f"  [Tool] Result: {result}")

        # ====================================================================
        # RETURN SUCCESS RESULT
        # ====================================================================
        return {
            "success": True,
            "expression": expression,
            "result": float(result) if isinstance(result, (int, float, np.number)) else str(result),
            "type": type(result).__name__
        }

    except Exception as e:
        # ====================================================================
        # HANDLE ERRORS (INVALID EXPRESSION, UNDEFINED FUNCTION, ETC.)
        # ====================================================================
        print(f"  [Error] Calculation failed: {e}")
        return {
            "success": False,
            "error": f"Calculation failed: {str(e)}",
            "expression": expression,
            "help": "Use standard math operations (+, -, *, /, **) and functions (sqrt, log, sin, cos, mean, etc.)"
        }

print("✓ Created calculator tool with safe evaluation")
print("  - Supports: Basic math, scientific functions, statistics")
print("  - Security: Restricted eval with whitelisted functions only")
print("  - Usage: calculate('2 + 2'), calculate('sqrt(16)'), calculate('mean([1,2,3])')")

✓ Created calculator tool with safe evaluation
  - Supports: Basic math, scientific functions, statistics
  - Security: Restricted eval with whitelisted functions only
  - Usage: calculate('2 + 2'), calculate('sqrt(16)'), calculate('mean([1,2,3])')


In [ ]:
# Create Multi-Tool Research Agent with Google ADK

research_agent = Agent(
    name="research_assistant",
    model="gemini-2.5-flash",  # Google ADK uses Gemini models
    instruction="""You are an advanced research assistant with access to multiple tools, if you don't have capability to do your your own knowledge like grouping of data etc:

1. **download_and_index_file**: Download PDFs, CSVs, or text files from the internet and index them
   - Automatically generates Vertex AI embeddings
   - Adds to FAISS vector index
   - Stores in local document store

2. **search_documents**: Search through downloaded documents using semantic vector search
   - Uses FAISS for fast k-NN similarity search
   - Returns semantically similar documents (not just keyword matches)
   - Provides similarity scores and relevant excerpts

3. **web_search**: Search the web for current information and recent developments
   - Simulated for demo (ready for Google Search API integration)

4. **calculate**: Perform mathematical calculations and quantitative analysis
   - Supports standard operations and numpy functions

Your workflow:
- When asked about a specific document or dataset, use download_and_index_file to get it
- Use search_documents to find relevant information in downloaded files using semantic search
- Use web_search for current events or general information not in documents
- Use calculate for any numerical analysis, statistics, or math operations
- Combine information from multiple sources to provide comprehensive answers
- Always cite sources (document IDs, URLs, page numbers)

Be thorough, accurate, and cite all sources. When analyzing data, provide clear insights.""",
    tools=[download_and_index_file, search_documents, web_search, calculate]
)

print("✓ Created comprehensive multi-tool research agent with Google ADK")
print(f"  Agent Name: {research_agent.name}")
print(f"  Model: {research_agent.model}")
print(f"  Tools: {len(research_agent.tools)} tools")
print("\n  Tool Capabilities:")
print("    1. download_and_index_file → Vertex AI Embeddings → FAISS Index")
print("    2. search_documents → FAISS Vector Search → Semantic Results")
print("    3. web_search → External Knowledge")
print("    4. calculate → Mathematical Analysis")
print("\n  Integration:")
print("    ✓ Google ADK Agent Framework")
print("    ✓ Vertex AI text-embedding-005")
print("    ✓ FAISS IndexFlatL2 (L2 distance)")
print("    ✓ Local file system storage")

✓ Created comprehensive multi-tool research agent with Google ADK
  Agent Name: research_assistant
  Model: gemini-2.5-flash
  Tools: 4 tools

  Tool Capabilities:
    1. download_and_index_file → Vertex AI Embeddings → FAISS Index
    2. search_documents → FAISS Vector Search → Semantic Results
    3. web_search → External Knowledge
    4. calculate → Mathematical Analysis

  Integration:
    ✓ Google ADK Agent Framework
    ✓ Vertex AI text-embedding-005
    ✓ FAISS IndexFlatL2 (L2 distance)
    ✓ Local file system storage


In [ ]:
# ============================================================================
# IMPORTS: Understanding Google ADK Components
# ============================================================================

# Python Standard Libraries
import os                                    # Operating system functions (environment variables)
import asyncio                               # Asynchronous I/O (for async/await pattern)
import json                                  # JSON encoding/decoding
from typing import Dict, Any, Optional       # Type hints for better code clarity

# ============================================================================
# GOOGLE ADK (Agentic Development Kit) - Core Components
# ============================================================================

from google.adk.agents import Agent
# Agent: The main class for creating AI agents
# Think of Agent as a "smart assistant" that can:
# - Understand natural language
# - Use tools (Python functions)
# - Make decisions about which tools to call
# - Maintain conversation context
#
# Example:
#   agent = Agent(
#       name="my_assistant",           # Friendly name for your agent
#       model="gemini-2.5-flash",  # Which LLM to use (brain of the agent)
#       instruction="You are helpful",  # System prompt (personality/behavior)
#       tools=[function1, function2]   # Python functions the agent can call
#   )

from google.adk.runners import Runner
# Runner: Executes the agent and manages the conversation flow
# Think of Runner as the "engine" that:
# - Takes user messages
# - Sends them to the agent
# - Handles tool calls
# - Streams back responses
# - Manages the conversation loop
#
# The Runner coordinates between:
#   User → Agent → LLM → Tools → Response → User

from google.adk.sessions import InMemorySessionService
# InMemorySessionService: Stores conversation history in memory
# Think of this as the agent's "memory bank" that:
# - Stores all messages in a conversation
# - Keeps track of multiple conversations (sessions)
# - Maintains context between messages
# - Enables multi-turn conversations
#
# "InMemory" means:
# - Data stored in RAM (fast, but lost when program stops)
# - Good for development and testing
# - For production, use database-backed session service
#
# What's a Session?
# - A session is a conversation thread
# - Like a chat thread in a messaging app
# - Each session has a unique ID
# - Different sessions are completely separate

from google.genai import types
# types: Data structures for messages and content
# Provides classes for:
# - Content: A message (user or agent)
# - Part: A piece of content (text, image, etc.)
# - These are the building blocks of conversations
#
# Example message structure:
#   Content(
#       role="user",                    # Who sent it: "user" or "model"
#       parts=[Part(text="Hello!")]     # What they said
#   )

print("✓ All imports successful!")
print("\nWhat we imported:")
print("  • Agent          - Create AI agents")
print("  • Runner         - Execute agents")
print("  • SessionService - Store conversation history")
print("  • types          - Message data structures")

✓ All imports successful!

What we imported:
  • Agent          - Create AI agents
  • Runner         - Execute agents
  • SessionService - Store conversation history
  • types          - Message data structures


In [ ]:
# ============================================================================
# HELPER FUNCTION TO RUN AGENTS WITH CONVERSATION MEMORY
# ============================================================================

# Global session service to maintain conversation history
# IMPORTANT: Reusing the same session_service enables conversation memory!
global_session_service = InMemorySessionService()

async def run_agent(agent: Agent, message: str, user_id: str = "user1", session_id: str = "session1"):
    """
    Run an agent with a message and print the response.

    IMPORTANT for conversation memory:
    - Same session_id = remembers previous messages
    - Different session_id = separate conversation

    Example:
        await run_agent(agent, "My name is Alice", session_id="conv1")
        await run_agent(agent, "What's my name?", session_id="conv1")  # Remembers "Alice"
        await run_agent(agent, "Hi", session_id="conv2")  # Separate conversation

    Args:
        agent: The agent to run
        message: User message
        user_id: User identifier (default: "user1")
        session_id: Session identifier (default: "session1")
    """
    global global_session_service

    APP_NAME = "adk_tutorial"

    # ====================================================================
    # STEP 1: CREATE SESSION (Required by ADK!)
    # ====================================================================
    # The session MUST be created before calling run_async()
    # If session already exists, this will fail silently (which is fine)
    try:
        session = await global_session_service.create_session(
            app_name=APP_NAME,
            user_id=user_id,
            session_id=session_id
        )
    except Exception:
        # Session already exists - that's fine, we'll reuse it
        pass

    # ====================================================================
    # STEP 2: CREATE RUNNER
    # ====================================================================
    # IMPORTANT: app_name must match the session's app_name!
    runner = Runner(
        agent=agent,
        app_name=APP_NAME,
        session_service=global_session_service
    )

    # ====================================================================
    # STEP 3: CREATE MESSAGE
    # ====================================================================
    message_content = types.Content(
        role="user",
        parts=[types.Part(text=message)]
    )

    # ====================================================================
    # STEP 4: PRINT USER MESSAGE
    # ====================================================================
    print(f"\n{'='*70}")
    print(f"User [{session_id}]: {message}")
    print(f"{'='*70}")
    print("Agent: ", end="", flush=True)

    # ====================================================================
    # STEP 5: RUN AGENT
    # ====================================================================
    # The session was created in Step 1, so run_async will find it
    full_response = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=message_content
    ):
        if hasattr(event, 'content') and event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, 'text') and part.text:
                    print(part.text, end="", flush=True)
                    full_response += part.text

    print("\n")
    return full_response

print("✓ Helper function defined with proper session management")
print("  - Creates session before run_async (required by ADK)")
print("  - Same session_id = remembers conversation")
print("  - Different session_id = separate conversations")


✓ Helper function defined with proper session management
  - Creates session before run_async (required by ADK)
  - Same session_id = remembers conversation
  - Different session_id = separate conversations


In [ ]:
# Example 1: Download and analyze a CSV file

print("\\n" + "="*70)
print("EXAMPLE 1: Analyzing Data from a CSV File")
print("="*70 + "\\n")

# First, let's use a sample CSV URL (you can replace with any real CSV URL)
csv_url = "https://raw.githubusercontent.com/datasets/covid-19/master/data/countries-aggregated.csv"

await run_agent(
    research_agent,
    f"Please download and analyze the COVID-19 dataset from this URL: {csv_url}. Tell me what data it contains.",
    user_id="researcher1"
)



\n======================================================================
EXAMPLE 1: Analyzing Data from a CSV File
======================================================================\n

User [session1]: Please download and analyze the COVID-19 dataset from this URL: https://raw.githubusercontent.com/datasets/covid-19/master/data/countries-aggregated.csv. Tell me what data it contains.
Agent:   [Tool] Downloading file from: https://raw.githubusercontent.com/datasets/covid-19/master/data/countries-aggregated.csv
I'm here in detect file
  [Tool] Processing CSV file...
  [Tool] Generating semantic embedding with Vertex AI...
  [Tool] Successfully indexed document 'covid_data' (1187 characters)
  [Tool] Added to FAISS index at position 1
The COVID-19 dataset from the provided URL has been successfully downloaded and indexed.

It contains the following columns:
*   **Date**: The date of the data entry.
*   **Country**: The country to which the data belongs.
*   **Confirmed**: The number o

"The COVID-19 dataset from the provided URL has been successfully downloaded and indexed.\n\nIt contains the following columns:\n*   **Date**: The date of the data entry.\n*   **Country**: The country to which the data belongs.\n*   **Confirmed**: The number of confirmed COVID-19 cases.\n*   **Recovered**: The number of recovered COVID-19 cases.\n*   **Deaths**: The number of COVID-19 related deaths.\n\nHere's a summary of the numerical data:\n\n**Confirmed Cases:**\n*   Minimum: 0\n*   Maximum: 80,625,120\n*   Mean: 736,156.93\n*   Standard Deviation: 3,578,884.22\n*   25th Percentile: 1,220\n*   50th Percentile (Median): 23,692\n*   75th Percentile: 255,842\n\n**Recovered Cases:**\n*   Minimum: 0\n*   Maximum: 30,974,748\n*   Mean: 145,396.71\n*   Standard Deviation: 974,827.51\n*   25th Percentile: 0\n*   50th Percentile (Median): 126\n*   75th Percentile: 17,972.25\n\n**Deaths:**\n*   Minimum: 0\n*   Maximum: 988,609\n*   Mean: 13,999.44\n*   Standard Deviation: 59,113.58\n*   25th

In [ ]:
# Example 2: Download and search a PDF document

print("\\n" + "="*70)
print("EXAMPLE 2: Downloading and Searching PDF Documents")
print("="*70 + "\\n")

# Example with a sample PDF (replace with any accessible PDF URL)
# Using a sample research paper or technical document
pdf_url = "https://www.irs.gov/pub/irs-drop/n-25-67.pdf"  # "Attention is All You Need" paper

await run_agent(
    research_agent,
    f"Download this paper: {pdf_url} and tell me what it's about. Save it with doc_id 'irs'.",
    user_id="researcher1"
)


\n======================================================================
EXAMPLE 2: Downloading and Searching PDF Documents
======================================================================\n

User [session1]: Download this paper: https://www.irs.gov/pub/irs-drop/n-25-67.pdf and tell me what it's about. Save it with doc_id 'irs'.
Agent:   [Tool] Downloading file from: https://www.irs.gov/pub/irs-drop/n-25-67.pdf
I'm here in detect file
  [Tool] Processing PDF file...
  [Tool] Generating semantic embedding with Vertex AI...
  [Tool] Successfully indexed document 'irs' (15039 characters)
  [Tool] Added to FAISS index at position 2
  [Tool] Performing vector search for: 'What is this document about?'
  [Tool] Generating query embedding...
  [Tool] Found document with similarity score: 0.4911


CancelledError: 

In [ ]:
# Example 3: Combining web search, documents, and calculations

print("\\n" + "="*70)
print("EXAMPLE 3: Multi-Tool Research Query")
print("="*70 + "\\n")

await run_agent(
    research_agent,
    """I need to research machine learning trends. Can you:
    1. Search the web for recent ML developments
    2. Search our downloaded documents for relevant technical information
    3. Calculate the growth rate if ML adoption increased from 35% to 67% over 3 years""",
    user_id="researcher1"
)

# Example 4: Pure calculation
print("\\n" + "="*70)
print("EXAMPLE 4: Complex Calculations")
print("="*70 + "\\n")

await run_agent(
    research_agent,
    "Calculate the compound annual growth rate (CAGR) formula: ((Final/Initial)^(1/Years) - 1) * 100, where Final=150000, Initial=50000, Years=5",
    user_id="researcher1"
)

\n======================================================================
EXAMPLE 3: Multi-Tool Research Query
======================================================================\n

User [session1]: I need to research machine learning trends. Can you:
    1. Search the web for recent ML developments
    2. Search our downloaded documents for relevant technical information
    3. Calculate the growth rate if ML adoption increased from 35% to 67% over 3 years
Agent:   [Tool] Searching web for: 'recent machine learning developments'
  [Tool] Found 5 web results (simulated)
Here are some recent machine learning developments based on a web search:

*   **Research Journal**: Latest research and insights about recent machine learning developments. (Source: https://example.com/article-1)
*   **Expert Blog**: A comprehensive guide on recent machine learning developments, including best practices, expert recommendations, and practical applications. (Source: https://example.com/guide)
*   **Ind

_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}